# BERT-QPP data exploration — MS MARCO dev-small

Read-only EDA over the same data `BERTQPP_train_MSMARCO_dev_colab.ipynb` trains on: the MS MARCO **dev-small**
queries, their qrels, the full passage **collection**, and the official **BM25 top-1000** retrieved-docs run.
Nothing here is fed into training — it's just to look at what's actually in these files before trusting a
pipeline built on top of them.

Covers:
- **`queries.dev.small.tsv`** — query text, length distribution, duplicate/empty checks
- **`qrels.dev.small.tsv`** — relevance label distribution, judgments-per-query
- **`collection.tsv`** — the full ~8.8M-passage MS MARCO collection (streamed, not loaded into memory at once;
  passage-length stats come from a reservoir sample)
- **`top1000.dev.tsv`** — the official BM25 top-1000 run for the dev-small queries, plus how well it covers the
  qrels-judged relevant docs (recall@1000, rank-of-relevant-doc distribution)

**Assumes the data is already downloaded** and cached on Drive under `DATA_DIR` below (same layout the training
notebook produces: `collection.tsv`, `queries.dev.small.tsv`, `qrels.dev.small.tsv`, `top1000.dev.tsv`). This
notebook only reads those files -- it doesn't fetch or write anything.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pandas numpy matplotlib

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/precise-qpp"
DATA_DIR = f"{DRIVE_ROOT}/data"

COLLECTION_PATH = f"{DATA_DIR}/collection.tsv"
DEV_QUERIES_PATH = f"{DATA_DIR}/queries.dev.small.tsv"
DEV_QRELS_PATH = f"{DATA_DIR}/qrels.dev.small.tsv"
TOP1000_RUN_PATH = f"{DATA_DIR}/top1000.dev.tsv"

REQUIRED_PATHS = [COLLECTION_PATH, DEV_QUERIES_PATH, DEV_QRELS_PATH, TOP1000_RUN_PATH]
missing = [p for p in REQUIRED_PATHS if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Missing expected file(s): {missing}. This notebook assumes the data is already downloaded -- "
        f"update the paths above if it lives somewhere else on Drive."
    )

for path in REQUIRED_PATHS:
    print(f"{path}: {os.path.getsize(path) / 1e6:,.1f} MB")

## Queries — `queries.dev.small.tsv`

In [ ]:
import pandas as pd

queries_df = pd.read_csv(DEV_QUERIES_PATH, sep="\t", names=["qid", "query"], dtype={"qid": str})
print(f"[INFO] {len(queries_df):,} dev queries")
queries_df.head()

In [ ]:
queries_df["n_words"] = queries_df["query"].str.split().str.len()
queries_df["n_chars"] = queries_df["query"].str.len()

print(queries_df[["n_words", "n_chars"]].describe())

n_empty = (queries_df["query"].str.strip() == "").sum()
n_dup_qid = queries_df["qid"].duplicated().sum()
n_dup_text = queries_df["query"].duplicated().sum()
print(f"[CHECK] empty queries: {n_empty}")
print(f"[CHECK] duplicate qids: {n_dup_qid}")
print(f"[CHECK] duplicate query text (different qid, same text): {n_dup_text}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(queries_df["n_words"], bins=range(0, int(queries_df["n_words"].max()) + 2))
axes[0].set_title("Query length (words)")
axes[0].set_xlabel("words")
axes[1].hist(queries_df["n_chars"], bins=50)
axes[1].set_title("Query length (chars)")
axes[1].set_xlabel("chars")
plt.tight_layout()
plt.show()

In [ ]:
queries_df.sample(10, random_state=0)[["qid", "query"]]

## Qrels — `qrels.dev.small.tsv`

In [ ]:
qrels_df = pd.read_csv(
    DEV_QRELS_PATH, sep=r"\s+", names=["qid", "iter", "docid", "rel"], dtype={"qid": str, "docid": str}
)
print(f"[INFO] {len(qrels_df):,} qrel rows")
qrels_df.head()

In [ ]:
print("[CHECK] relevance label distribution:")
print(qrels_df["rel"].value_counts().sort_index())

judgments_per_query = qrels_df.groupby("qid").size()
print("\n[CHECK] judged docs per query:")
print(judgments_per_query.describe())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(judgments_per_query, bins=range(1, int(judgments_per_query.max()) + 2))
ax.set_title("Judged docs per query (dev qrels)")
ax.set_xlabel("# judged docs")
plt.show()

In [ ]:
judged_qids = set(qrels_df["qid"])
query_qids = set(queries_df["qid"])
overlap = judged_qids & query_qids
print(f"[CHECK] {len(overlap):,} / {len(query_qids):,} dev queries have >=1 qrel ({len(overlap) / len(query_qids):.1%})")

## Collection — `collection.tsv`

~8.8M passages / ~2.9GB — too large to load into a single DataFrame comfortably, so this streams the file once:
full counts (passages, empty passages) are exact, but per-passage length stats/plots come from a bounded
reservoir sample (`RESERVOIR_SIZE`) so memory stays flat regardless of collection size.

In [ ]:
import random

random.seed(0)
RESERVOIR_SIZE = 200_000

reservoir_words, reservoir_chars = [], []
n_passages = 0
n_empty_passages = 0
sample_rows = []

with open(COLLECTION_PATH) as f:
    for line in f:
        docid, text = line.rstrip("\n").split("\t", 1)
        n_words = len(text.split())
        n_chars = len(text)
        if not text.strip():
            n_empty_passages += 1

        if n_passages < RESERVOIR_SIZE:
            reservoir_words.append(n_words)
            reservoir_chars.append(n_chars)
        else:
            j = random.randint(0, n_passages)
            if j < RESERVOIR_SIZE:
                reservoir_words[j] = n_words
                reservoir_chars[j] = n_chars

        if n_passages < 5:
            sample_rows.append((docid, text))

        n_passages += 1

print(f"[INFO] {n_passages:,} passages in collection.tsv")
print(f"[CHECK] empty passages: {n_empty_passages:,}")
for docid, text in sample_rows:
    print(f"  {docid}: {text[:150]}")

In [ ]:
import numpy as np

reservoir_words = np.array(reservoir_words)
reservoir_chars = np.array(reservoir_chars)

print(f"[INFO] passage length stats (sample of {len(reservoir_words):,} / {n_passages:,} passages):")
print(pd.Series(reservoir_words, name="n_words").describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(reservoir_words, bins=50)
axes[0].set_title("Passage length (words, sampled)")
axes[1].hist(reservoir_chars, bins=50)
axes[1].set_title("Passage length (chars, sampled)")
plt.tight_layout()
plt.show()

## BM25 top-1000 retrieved docs — `top1000.dev.tsv`

Rows are in BM25 rank order within each query's block (the file has no explicit rank column), same assumption
`create_map.py`'s callers rely on -- rank here is just each row's position within its query.

In [ ]:
from collections import defaultdict

rank_counters = defaultdict(int)
run_ranks = defaultdict(dict)  # qid -> {docid: rank}
seen_pids = set()
n_run_rows = 0
sample_run_rows = []

with open(TOP1000_RUN_PATH) as f:
    for line in f:
        qid, pid, query_text, passage_text = line.rstrip("\n").split("\t")
        rank_counters[qid] += 1
        rank = rank_counters[qid]
        run_ranks[qid][pid] = rank
        seen_pids.add(pid)
        n_run_rows += 1
        if n_run_rows <= 3:
            sample_run_rows.append((qid, pid, rank, query_text, passage_text[:100]))

print(f"[INFO] {n_run_rows:,} rows across {len(rank_counters):,} queries in top1000.dev.tsv")
print(f"[INFO] {len(seen_pids):,} unique passages retrieved")
for row in sample_run_rows:
    print(row)

In [ ]:
docs_per_query = pd.Series(rank_counters, name="n_docs")
print(docs_per_query.describe())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(docs_per_query, bins=50)
ax.set_title("BM25 docs retrieved per query (top-1000 run)")
ax.set_xlabel("# docs retrieved")
plt.show()

## Cross-cutting: does the BM25 run actually cover the relevant docs?

For every qrels-judged **relevant** (`rel > 0`) doc whose query also appears in the BM25 run, look up its rank
in that run. Missing entirely means it fell outside the top-1000 -- i.e. BM25 recall@1000 for that judgment.

In [ ]:
rel_qrels = qrels_df[qrels_df["rel"] > 0]

ranks_of_relevant = []
n_missed = 0
n_checked = 0

for qid, group in rel_qrels.groupby("qid"):
    if qid not in run_ranks:
        continue
    for docid in group["docid"]:
        n_checked += 1
        rank = run_ranks[qid].get(docid)
        if rank is None:
            n_missed += 1
        else:
            ranks_of_relevant.append(rank)

print(f"[CHECK] {n_checked:,} relevant (rel>0) qrel judgments for queries present in the BM25 run")
print(f"[CHECK] {len(ranks_of_relevant):,} found within top-1000 ({len(ranks_of_relevant) / n_checked:.1%} recall@1000)")
print(f"[CHECK] {n_missed:,} relevant docs NOT in the BM25 top-1000 run")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ranks_of_relevant, bins=50)
ax.set_title("Rank of relevant docs within the BM25 top-1000 run")
ax.set_xlabel("rank")
plt.show()

print(pd.Series(ranks_of_relevant, name="rank").describe())

## Next steps

This is read-only exploration -- nothing here writes back to Drive. To actually train on this data, see
`BERTQPP_train_MSMARCO_dev_colab.ipynb`, which derives its own BM25 run/collection subset from `top1000.dev.tsv`
rather than downloading the full `collection.tsv` this notebook used.